# Titanic으로 배우는 머신러닝 Workflow

이 노트북의 목적은 단순히 높은 Kaggle 점수를 만드는 것이 아니라,  
**머신러닝 문제를 어떤 순서로 풀어야 하는지 이해하는 것**이다.

핵심 흐름:

1. 문제 정의
2. 데이터 구조 확인
3. 검증 방법 고정
4. 가장 단순한 baseline 만들기
5. feature를 하나씩 추가하며 비교
6. EDA로 새로운 가설 세우기
7. feature engineering
8. ablation으로 추가 가치 검증
9. 최종 feature 선택
10. 다른 모델과 비교
11. 전체 train으로 학습 후 test 예측

## 0. 이 문제는 무엇인가?

Titanic 문제는 정답 `Survived`가 이미 주어진 **지도학습(supervised learning)** 문제이다.

`Survived`는 두 값만 가진다.

- `0`: 사망
- `1`: 생존

따라서 이 문제는 **이진 분류(binary classification)** 문제이다.

평가 지표는 accuracy이다.

\[
\text{Accuracy}
=
\frac{\text{맞게 예측한 사람 수}}{\text{전체 사람 수}}
\]

## 1. 라이브러리와 데이터 불러오기

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
candidate_dirs = [
    Path("/kaggle/input/titanic"),
    Path("/kaggle/input/competitions/titanic"),
    Path.cwd(),
    Path.cwd() / "data",
    Path("/mnt/data"),
]

data_dir = None

for folder in candidate_dirs:
    if (
        (folder / "train.csv").exists()
        and (folder / "test.csv").exists()
    ):
        data_dir = folder
        break

if data_dir is None:
    raise FileNotFoundError(
        "train.csv와 test.csv를 찾지 못했습니다.\n"
        "두 파일을 현재 노트북과 같은 폴더 또는 data 폴더에 넣어주세요."
    )

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

print("데이터 폴더:", data_dir)
print("train shape:", train.shape)
print("test shape:", test.shape)

train.head()

### train과 test의 차이

`train`에는 정답 열 `Survived`가 있고, `test`에는 없다.

- train: 모델 학습과 validation에 사용
- test: 최종 예측에만 사용

In [ ]:
print("train에만 있는 열:", set(train.columns) - set(test.columns))
print("test에만 있는 열:", set(test.columns) - set(train.columns))

## 2. 데이터 구조 확인

In [ ]:
train.info()

In [ ]:
data_summary = pd.DataFrame({
    "dtype": train.dtypes,
    "missing_count": train.isna().sum(),
    "missing_rate": train.isna().mean(),
    "nunique": train.nunique()
})

data_summary.sort_values("missing_rate", ascending=False)

### Recall

- 어떤 열에 결측치가 많은가?
- 수치형 변수와 범주형 변수는 각각 무엇인가?
- `PassengerId`는 예측 정보일까, 단순 식별자일까?

## 3. 검증 방법 고정

test에는 정답이 없으므로 test accuracy를 직접 계산할 수 없다.

그래서 train을 다시 두 부분으로 나눈다.

- training set: 모델 학습
- validation set: 성능 평가

이후 모든 실험에서 같은 split을 사용한다.

In [ ]:
y = train["Survived"]

train_idx, valid_idx = train_test_split(
    train.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("전체 생존률:", round(y.mean(), 4))
print("train 생존률:", round(y.loc[train_idx].mean(), 4))
print("valid 생존률:", round(y.loc[valid_idx].mean(), 4))

### 왜 split을 고정하는가?

feature 비교 중 validation에 들어가는 승객이 바뀌면  
성능 차이가 feature 때문인지, 데이터 분할 차이 때문인지 알기 어렵다.

따라서 다음을 고정한다.

- train/validation split
- model
- 평가 지표
- random_state

그리고 feature만 바꾼다.

## 4. 가장 단순한 baseline

baseline은 완성된 모델이 아니라 **기준점**이다.

처음부터 복잡한 파생변수를 넣지 않고,  
가장 명확하고 해석하기 쉬운 변수부터 시작한다.

첫 baseline은 `Sex` 하나만 사용한다.

In [ ]:
df = train.copy()

### 공통 평가 함수

In [ ]:
def evaluate_logistic_model(
    data,
    features,
    numeric_features=None,
    categorical_features=None,
    model_name="model"
):
    numeric_features = numeric_features or []
    categorical_features = categorical_features or []

    X = data[features]

    X_train = X.loc[train_idx]
    X_valid = X.loc[valid_idx]
    y_train = y.loc[train_idx]
    y_valid = y.loc[valid_idx]

    transformers = []

    if numeric_features:
        transformers.append(
            ("num", StandardScaler(), numeric_features)
        )

    if categorical_features:
        transformers.append(
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_features
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

    model.fit(X_train, y_train)

    pred = model.predict(X_valid)
    accuracy = accuracy_score(y_valid, pred)

    print(f"{model_name}: {accuracy:.4f}")

    return model, accuracy

In [ ]:
experiment_results = []

def run_experiment(
    name,
    features,
    numeric_features=None,
    categorical_features=None
):
    model, accuracy = evaluate_logistic_model(
        data=df,
        features=features,
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        model_name=name
    )

    experiment_results.append({
        "experiment": name,
        "features": ", ".join(features),
        "validation_accuracy": accuracy
    })

    return model

In [ ]:
model_1 = run_experiment(
    name="Model 1: Sex only",
    features=["Sex"],
    categorical_features=["Sex"]
)

### 해석

이 점수는 이후 모든 feature 실험의 가장 단순한 기준점이다.

여기서 질문:

> 성별 정보만으로도 생존 여부를 어느 정도 예측할 수 있는가?

## 5. Pclass 추가

이제 `Pclass`를 추가한다.

가설:

> 객실 등급이 생존 가능성과 관련이 있을 수 있다.

이번 실험에서는 모델과 split은 그대로 두고 feature만 바꾼다.

In [ ]:
model_2 = run_experiment(
    name="Model 2: Sex + Pclass",
    features=["Sex", "Pclass"],
    numeric_features=["Pclass"],
    categorical_features=["Sex"]
)

### 비교 질문

- `Sex only`보다 점수가 올랐는가?
- 올랐다면 `Pclass`가 Sex에 없는 추가 정보를 제공했다고 볼 수 있는가?
- 차이가 매우 작다면 한 번의 split만으로 확정할 수 있는가?

## 6. 가족 변수 탐색

원본 데이터에는 가족 관련 변수가 두 개 있다.

- `SibSp`: 함께 탑승한 형제자매·배우자 수
- `Parch`: 함께 탑승한 부모·자녀 수

먼저 원본 변수 자체를 추가해본다.

In [ ]:
model_3 = run_experiment(
    name="Model 3: Sex + Pclass + SibSp + Parch",
    features=["Sex", "Pclass", "SibSp", "Parch"],
    numeric_features=["Pclass", "SibSp", "Parch"],
    categorical_features=["Sex"]
)

### FamilySize 만들기

`SibSp`와 `Parch`는 둘 다 동행 가족 수를 나타낸다.

따라서 다음 파생변수를 만든다.

\[
\text{FamilySize}
=
\text{SibSp}
+
\text{Parch}
+
1
\]

마지막 `1`은 본인을 포함하기 위한 값이다.

In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

df[
    ["SibSp", "Parch", "FamilySize"]
].head()

In [ ]:
model_4 = run_experiment(
    name="Model 4: Sex + Pclass + FamilySize",
    features=["Sex", "Pclass", "FamilySize"],
    numeric_features=["Pclass", "FamilySize"],
    categorical_features=["Sex"]
)

### IsAlone 만들기

가족 수가 정확히 몇 명인지보다 혼자인지 아닌지가 더 중요한 정보일 수도 있다.

\[
\text{IsAlone}
=
\begin{cases}
1, & \text{FamilySize}=1 \\
0, & \text{otherwise}
\end{cases}
\]

In [ ]:
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df[
    ["FamilySize", "IsAlone"]
].head()

In [ ]:
model_5 = run_experiment(
    name="Model 5: Sex + Pclass + FamilySize + IsAlone",
    features=["Sex", "Pclass", "FamilySize", "IsAlone"],
    numeric_features=["Pclass", "FamilySize", "IsAlone"],
    categorical_features=["Sex"]
)

In [ ]:
family_comparison = (
    pd.DataFrame(experiment_results)
      .sort_values("validation_accuracy", ascending=False)
      .reset_index(drop=True)
)

family_comparison

### 가족 feature 해석

다음 질문에 답한다.

- `SibSp + Parch`와 `FamilySize` 중 어느 쪽이 더 좋았는가?
- `IsAlone`을 추가했을 때 성능이 더 좋아졌는가?
- `FamilySize`와 `IsAlone`은 중복 정보를 포함하는가?
- 이후 Cabin 실험의 baseline으로 어떤 feature 조합을 선택할 것인가?

여기서 가장 합리적인 가족 feature 조합을 **새 baseline**으로 선택한다.

## 7. Cabin EDA와 가설 수립

이제 Cabin 정보를 바로 모델에 넣지 않고 먼저 EDA를 한다.

가설:

> Cabin의 기록 여부 또는 물리적 위치가 기존 baseline에 없는 추가 정보를 제공할 수 있다.

질문:

1. Cabin 정보가 있는 승객과 없는 승객의 생존률은 다른가?
2. Deck별 생존률은 다른가?
3. Deck과 Pclass는 얼마나 겹치는가?
4. 같은 Pclass 안에서도 차이가 남는가?

In [ ]:
df["CabinKnown"] = df["Cabin"].notna().astype(int)

cabin_known_summary = (
    df.groupby("CabinKnown")
      .agg(
          PassengerCount=("Survived", "count"),
          SurvivorCount=("Survived", "sum"),
          SurvivalRate=("Survived", "mean")
      )
      .reset_index()
)

cabin_known_summary

### 여러 객실 기록 확인

In [ ]:
multiple_cabins = df.loc[
    df["Cabin"].fillna("").str.contains(r"\s"),
    ["PassengerId", "Name", "Cabin"]
]

print("여러 객실이 기록된 행 수:", len(multiple_cabins))
multiple_cabins

여러 객실이 존재한다면 객실 번호 하나만 사용하는 것은 정보 손실을 만들 수 있다.

이번 단계에서는 먼저 Deck만 만든다.

In [ ]:
df["Deck"] = (
    df["Cabin"]
      .str.extract(r"^([A-Za-z])")[0]
      .fillna("Unknown")
)

df[["Cabin", "Deck"]].head(15)

In [ ]:
deck_summary = (
    df.groupby("Deck")
      .agg(
          PassengerCount=("Survived", "count"),
          SurvivalRate=("Survived", "mean")
      )
      .sort_values("SurvivalRate", ascending=False)
      .reset_index()
)

deck_summary

In [ ]:
pd.crosstab(
    df["Deck"],
    df["Pclass"],
    normalize="index"
).round(3)

In [ ]:
cabin_class_summary = (
    df.groupby(["Pclass", "CabinKnown"])
      .agg(
          PassengerCount=("Survived", "count"),
          SurvivalRate=("Survived", "mean")
      )
      .reset_index()
)

cabin_class_summary

In [ ]:
deck_class_summary = (
    df.groupby(["Pclass", "Deck"])
      .agg(
          PassengerCount=("Survived", "count"),
          SurvivalRate=("Survived", "mean")
      )
      .reset_index()
)

deck_class_summary

### EDA와 feature 검증의 차이

EDA의 질문:

> 관계가 보이는가?

Feature 검증의 질문:

> 기존 baseline에 추가했을 때 validation accuracy가 실제로 좋아지는가?

EDA에서 차이가 크게 보여도 기존 feature와 정보가 겹치면 모델 성능은 좋아지지 않을 수 있다.

## 8. Cabin feature ablation

아래 `selected_baseline_features`는 가족 변수 실험 결과를 보고 직접 선택한다.

예시로는 다음 조합을 넣어두었다.

- Sex
- Pclass
- FamilySize
- IsAlone

In [ ]:
selected_baseline_features = [
    "Sex",
    "Pclass",
    "FamilySize",
    "IsAlone"
]

selected_baseline_numeric = [
    "Pclass",
    "FamilySize",
    "IsAlone"
]

selected_baseline_categorical = [
    "Sex"
]

In [ ]:
cabin_experiments = []

def run_cabin_experiment(
    name,
    features,
    numeric_features,
    categorical_features
):
    model, accuracy = evaluate_logistic_model(
        data=df,
        features=features,
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        model_name=name
    )

    cabin_experiments.append({
        "experiment": name,
        "features": ", ".join(features),
        "validation_accuracy": accuracy
    })

    return model

In [ ]:
baseline_model = run_cabin_experiment(
    name="Cabin Baseline",
    features=selected_baseline_features,
    numeric_features=selected_baseline_numeric,
    categorical_features=selected_baseline_categorical
)

cabin_known_model = run_cabin_experiment(
    name="Baseline + CabinKnown",
    features=selected_baseline_features + ["CabinKnown"],
    numeric_features=selected_baseline_numeric + ["CabinKnown"],
    categorical_features=selected_baseline_categorical
)

deck_model = run_cabin_experiment(
    name="Baseline + Deck",
    features=selected_baseline_features + ["Deck"],
    numeric_features=selected_baseline_numeric,
    categorical_features=selected_baseline_categorical + ["Deck"]
)

cabin_deck_model = run_cabin_experiment(
    name="Baseline + CabinKnown + Deck",
    features=selected_baseline_features + ["CabinKnown", "Deck"],
    numeric_features=selected_baseline_numeric + ["CabinKnown"],
    categorical_features=selected_baseline_categorical + ["Deck"]
)

In [ ]:
cabin_comparison = (
    pd.DataFrame(cabin_experiments)
      .sort_values("validation_accuracy", ascending=False)
      .reset_index(drop=True)
)

cabin_comparison

## 9. 최종 정리

### 지금까지의 workflow

1. Sex only baseline
2. Pclass 추가
3. SibSp, Parch 추가
4. FamilySize 생성
5. IsAlone 생성
6. 가족 feature 비교 후 새로운 baseline 선정
7. Cabin EDA
8. CabinKnown과 Deck ablation
9. 최종 feature 선택

### 반드시 기록할 내용

- 어떤 feature를 추가했을 때 성능이 개선되었는가?
- EDA에서 보인 관계와 모델 성능이 일치했는가?
- 어떤 feature는 기존 변수와 정보가 겹쳤는가?
- 작은 성능 차이를 얼마나 신뢰할 수 있는가?

## 10. 다음 단계

현재는 한 번의 train/validation split으로 비교했다.

다음 단계:

1. 같은 feature 실험을 5-fold cross-validation으로 반복
2. 최종 feature 조합 결정
3. Logistic Regression과 다른 모델 비교
4. 최종 모델을 전체 train 데이터로 재학습
5. test 예측
6. submission.csv 생성